# 실제 86셀 진단 시뮬레이션
첫 파일럿: 86셀 중 80셀을 선택하여 8셀 팩 10개를 만들고 6셀은 미사용 후보로 둡니다.
이 노트북은 이미 저장한 결과부터 보여줍니다. 맨 아래 셀에서 같은 실험을 다시 실행할 수 있습니다.
수명 예측은 관측된 용량만 쓰는 단순 Bayesian linear regression이며, 학습된 최종 GP 모델이 아닙니다.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
assert (ROOT / 'configs/real86.json').is_file(), 'simulation 작업공간에서 열어주세요.'
config = json.loads((ROOT / 'configs/real86.json').read_text())
OUT = ROOT / 'results' / config['run_name']
config

## 첫 실험 결과
점수는 해당 검사 종료 상태의 실제 RUL로 최적으로 배정한 경우를 100점으로 합니다.
검사로 소비한 수명까지 반영한 실제 팩 수명 합계도 함께 확인합니다.
세 난수 시드는 같은 86셀의 반복이며, 독립 표본 3개나 일반화 성능을 의미하지 않습니다.

In [ ]:
summary = pd.read_csv(OUT / 'summary.csv')
display(summary.pivot(index='additional_cell_cycles', columns='policy', values='score_mean').round(2))
display(summary[summary.additional_cell_cycles == summary.additional_cell_cycles.max()].round(2))
display(Image(filename=str(OUT / 'real86_comparison.png')))

## 수명 예측모델의 상태
이번 95점 목표는 미달성입니다. 의사결정 가치 근사가 불확실성 우선 방법보다 우수하다는 결과도 얻지 못했습니다.
추가 검사 전, 현재 용량만 정렬한 구성은 65.03점입니다. 실제 최종 성능 합계도 추가 검사 후 개선되지 않았으므로, 현 모델과 정책으로 검사 절감 성과를 주장할 수 없습니다.
명목 80% 예측구간의 실제 포함률도 결과표에 기록했습니다. 이 모델의 불확실성은 보정되지 않았습니다.

In [ ]:
metrics = pd.read_csv(OUT / 'metrics.csv')
display(metrics[metrics.additional_cell_cycles == 0][['capacity_sort_score_100', 'capacity_sort_utility_pack_cycles', 'oracle_current_pack_cycles']].head(1))
trace = pd.read_csv(OUT / 'inspection_trace.csv')
display(trace.head(10))
json.loads((OUT / 'manifest.json').read_text())

## 다시 실행하거나 조건 바꾸기
`configs/real86.json`에서 초기 관측량, 검사 예산, 팩 크기, 시나리오 수, 정책과 난수 시드를 바꿉니다. 다른 실험은 `run_name`을 바꿔 기존 결과와 구분하세요.
새 EOL 기준은 정답 파일을 함께 재계산하기 전에는 사용할 수 없습니다. 현재는 1.44 Ah 기준과 완전 관측되는 초기 101사이클 이내만 지원합니다.

In [ ]:
subprocess.run([sys.executable, '-m', 'battery_sim.real86'], cwd=ROOT, check=True)
print('실행 완료. 위 결과 셀을 다시 실행하면 새 결과가 표시됩니다.')